<a href="https://colab.research.google.com/github/vanecornejo/Procesos-Estocasticos/blob/main/Matriz%20fundamental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Matriz fundamental**

## *Serpientes y Escaleras*

Se analiza un tablero con casillas del 1 al 20. El jugador inicia en la casilla 1 y lanza un dado justo (valores de 1 a 6).

Existen:
- Escaleras:
  - 3 → 11
  - 15 → 19
- Serpientes:
  - 17 → 9
  - 13 → 5

El objetivo es llegar a la casilla 20, que es un estado absorbente.

Se desea calcular el número esperado de tiradas para terminar el juego.

*Analíticamente, Cadena de Markov*

Sea $E[i]$ el número esperado de tiradas para llegar a la casilla 20 desde la casilla $i$.

Se tiene:

- $E[20] = 0$ (estado absorbente)

Para $i < 20$:

$E[i] = 1 + \frac{1}{6} * \sum(E[f(i+k)])$ para $k = 1,...,6$

donde $f(x)$ representa el movimiento considerando serpientes y escaleras.

Esto genera un sistema de ecuaciones lineales.

El código de la solución analítica:

In [162]:
import numpy as np

In [163]:
n = 20  # Número de estados

# Diccionario de serpientes y escaleras
transiciones = {
    3: 11,
    15: 19,
    17: 9,
    13: 5
}

In [164]:
# Definimos una función para subir escaleras o bajar serpientes
# si el dado cae en las casillas 3, 15, 17 o 13
def mover(pos):
    return transiciones.get(pos, pos)

In [165]:
# Construimos el sistema A * E = b
A = np.zeros((19,19))
b = np.ones(19)

for i in range(1,20):  # estados 1 a 19
    A[i-1, i-1] = 1

    for k in range(1,7):
        j = i + k

        if j > 20:
            j = 20

        j = mover(j)

        if j != 20:
            A[i-1, j-1] -= 1/6

# Resolver sistema
E = np.linalg.solve(A, b)

# Imprimimos los resultados que buscamos
print("Esperanza desde cada estado:\n", E)
print("\nNúmero esperado de tiradas desde el inicio:", E[0])

Esperanza desde cada estado:
 [6.88063571 6.66901574 6.72237445 6.39022298 6.1131444  5.80862793
 5.85213029 5.39929597 4.77082512 4.39731419 4.45067289 3.98152915
 3.12933841 2.68229007 2.5156234  2.15624863 1.36111111 1.16666667
 1.        ]

Número esperado de tiradas desde el inicio: 6.880635705704065


Ahora vamos a simular el juego múltiples veces para estimar el número promedio de tiradas y verificar lo obtenido con el método analítico.

In [166]:
import random

# Creamos una función que haga la simulación del juego
def jugar():
    pos = 1
    tiros = 0

    while pos < 20:
        dado = random.randint(1,6)
        pos += dado

        if pos > 20:
            pos = 20

        pos = mover(pos)
        tiros += 1

    return tiros

In [167]:
# Llamamos la función del juego
N = 100000

resultados = [jugar() for _ in range(N)]

print("Promedio de tiradas", np.mean(resultados))

Promedio de tiradas 6.88453


El número esperado de tiradas para finalizar el juego es aproximadamente 6.

Todos los estados transientes conducen al estado absorbente, por lo que siempre termina el juego después de ciertos tiros.

## *Caminata del ratón*

Se tiene un grafo con estados del 0 al 8.

- Estado 7: comida (absorbente)
- Estado 8: shock (absorbente)

El ratón inicia en el estado 0 y se mueve aleatoriamente a uno de sus vecinos con igual probabilidad.

Vamos a calcular la probabilidad de que el ratón llegue a la comida y validar el resultado mediante simulación.

*Analíticamente:*

Sea $p[i]$ la probabilidad de llegar a la comida desde el estado $i$.

Condiciones:
- $p[7]$ = 1
- $p[8]$ = 0

Es decir, el estado inicial está en 0 y los estados absorbentes son 7 (Comida) y 8 (Shock).

Para otros estados:

$p[i]$ = promedio de $p[j]$ sobre vecinos $j$

Esto genera un sistema de ecuaciones lineales.

In [168]:
# vecinos del grafo
vecinos = {
    0: [1,2],
    1: [0,3],
    2: [0,3,8],
    3: [1,2,4,5],
    4: [3,6],
    5: [3,6,8],
    6: [4,5,7],
    7: [],
    8: []
}

# estados no absorbentes
estados = [0,1,2,3,4,5,6]

A = np.zeros((7,7))
b = np.zeros(7)

for i in estados:
    idx = estados.index(i)
    A[idx, idx] = 1

    for j in vecinos[i]:
        if j == 7:
            b[idx] += 1 / len(vecinos[i])
        elif j != 8:
            jdx = estados.index(j)
            A[idx, jdx] -= 1 / len(vecinos[i])

# resolver sistema
p = np.linalg.solve(A, b)

print("Probabilidades:", p)
print("Probabilidad desde el estado 0:", p[0])

Probabilidades: [0.19379845 0.23255814 0.15503876 0.27131783 0.41860465 0.27906977
 0.56589147]
Probabilidad desde el estado 0: 0.19379844961240306


Haciendo la simulación del juego varias veces:

In [169]:
def caminar():
    pos = 0

    while pos not in [7,8]:
        pos = random.choice(vecinos[pos])

    return pos == 7

N = 100000
resultados = [caminar() for _ in range(N)]

print("La probabilidad de que el ratón llegue a la comida iniciando en 0 es:", np.mean(resultados))

La probabilidad de que el ratón llegue a la comida iniciando en 0 es: 0.19451


La probabilidad de que el ratón alcance la comida iniciando en el estado 0 es aproximadamente 0.19.